In [1]:
import os
import torch 
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch.optim as optim

In [2]:
class RunOrWalkNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(6, 4)  # Input layer: 6 neurons, Hidden Layer: 4 neurons
        self.fc2 = nn.Linear(4, 2)  # Hidden layer: 4 neurons, Output Layer: 2 neurons


    def forward(self, x):
        x = self.fc1(x)
        x = F.leaky_relu(x)
        x = self.fc2(x)
        output = F.softmax(x, dim=1)  # dimension 1: softmax function applied along second dimension of the tensor
        return output

In [3]:
class RunOrWalkDataset(Dataset):
    def __init__(self, file_path):
        # preprocessing done here
        self.data = pd.read_csv(file_path)
        self.labels = self.data.pop('activity').values
        self.data = self.data.drop(columns=['date', 'time', 'username', 'wrist'])
        # L2 Normalisation for each column
        self.data = self.data / np.linalg.norm(self.data, axis=0)
        
        # Scale to same range
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.data)
        min_max_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_min_max_data = min_max_scaler.fit_transform(scaled_data)
        self.data = pd.DataFrame(scaled_min_max_data, columns=self.data.columns)


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = torch.tensor(self.data.iloc[idx].values, dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)  # Assuming labels are integers
        return features, label


In [4]:
dataset = RunOrWalkDataset("./dataset.csv")
train_data, test_data = train_test_split(dataset, test_size=0.25, random_state=1334)  # my favourite number

In [5]:
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


In [6]:
model = RunOrWalkNN()
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

In [7]:
epochs = 50

for epoch in range(epochs):
    model.train()  # Sets the model to training mode
    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

    model.eval()  # Sets the model to evaluation mode
    with torch.no_grad():  # Disable gradient calculation during testing
        correct = 0
        total = 0
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # we dont need all the maximum of a tensor, hence _
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    print(f'Epoch [{epoch + 1}/{epochs}], Test Accuracy: {accuracy:.2%}')


Epoch [1/50], Test Accuracy: 86.00%
Epoch [2/50], Test Accuracy: 88.44%
Epoch [3/50], Test Accuracy: 92.46%
Epoch [4/50], Test Accuracy: 94.39%
Epoch [5/50], Test Accuracy: 94.72%
Epoch [6/50], Test Accuracy: 94.88%
Epoch [7/50], Test Accuracy: 95.13%
Epoch [8/50], Test Accuracy: 94.92%
Epoch [9/50], Test Accuracy: 95.19%
Epoch [10/50], Test Accuracy: 95.39%
Epoch [11/50], Test Accuracy: 95.35%
Epoch [12/50], Test Accuracy: 95.64%
Epoch [13/50], Test Accuracy: 94.49%
Epoch [14/50], Test Accuracy: 95.62%
Epoch [15/50], Test Accuracy: 95.70%
Epoch [16/50], Test Accuracy: 95.50%
Epoch [17/50], Test Accuracy: 95.42%
Epoch [18/50], Test Accuracy: 95.89%
Epoch [19/50], Test Accuracy: 95.17%
Epoch [20/50], Test Accuracy: 95.86%
Epoch [21/50], Test Accuracy: 95.75%
Epoch [22/50], Test Accuracy: 95.63%
Epoch [23/50], Test Accuracy: 95.96%
Epoch [24/50], Test Accuracy: 95.77%
Epoch [25/50], Test Accuracy: 95.81%
Epoch [26/50], Test Accuracy: 96.23%
Epoch [27/50], Test Accuracy: 95.96%
Epoch [28/